In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# The Prince and the Tiger

Once upon a time, there was a king who wanted to marry his daughter to a prince. He had it proclaimed throughout the land that he was seeking a husband for his daughter. One day, a prince came by to apply for the position.

Since the king did not want to marry his daughter to some dullard, he led the prince into a room with nine doors. The king told the prince that the princess was in one of the rooms, but that there were other rooms behind which hungry tigers were waiting. Some rooms were also empty. If the prince opened a door with a tiger behind it, it would probably be his last mistake.

The king went on to say that there were signs on all the doors with statements on them. These statements behave as follows:

* In the rooms where there is a tiger, the statement on the sign is false.
* In the room where the princess is, the statement is true.
* With regards to the the empty rooms, the situation is a bit more complicated, because there are two possibilities:
    - Either **all** the inscriptions on the empty rooms are true,
    - or **all** the inscriptions on the empty rooms are false.

The prince then read the inscriptions. These were as follows:

1. Room: The princess is in a room with an odd number.
   There is no tiger in the rooms with an even number.
2. Room: This room is empty.
3. Room: The inscription on Room No. 5 is true, the inscription on Room No. 7 is false,
   and there is a tiger in Room No. 3.
4. Room: The inscription on Room No. 1 is false, there is no tiger in Room No. 8,
   and the inscription on Room No. 9 is true.
5. Room: If the inscription on Room No. 2 or on Room No. 4 is true,
   then there is no tiger in Room No. 1.
6. Room: The inscription on Room No. 3 is false, the princess is in Room No. 2,
   and there is no tiger in Room No. 2.
7. Room: The princess is in Room No. 1 and the inscription on Room No. 5 is true.
8. Room: There is no tiger in this room and Room No. 9 is empty.
9. Room: Neither in this room nor in Room No. 1 is there a tiger, and moreover, the
   inscription on Room No. 6 is true.

## Setting up the Required Notebooks

We import the notebook `07-Davis-Putnam-JW.ipynb`, which provides the function `solve` that implements the Davis-Putnam algorithm.  This notebook imports the notebook `04-CNF.ipynb`, which provides the function `normalize` that turns a formula into *conjunctive normal form*.  The notebook `04-CNF.ipynb` in turn imports the parser for propositional logic.

In [2]:
importNotebook "07-Davis-Putnam-JW.ipynb"

imported 

Propositional-Logic-Parser.ipynb, 04-CNF.ipynb, 07-Davis-Putnam-JW.ipynb

The function `parseKNF` takes one argument:
- `s` is a string that represents a formula from propositional logic.

It parses this string as a propositional formula and then turns this formula into a set of clauses, which is returned.  We have used this function already when discussing the Zebra problem.

In [3]:
let parseKNF (s: string) : CNF =
    normalize (parse s)

In [4]:
parseKNF "(p ∧ ¬q → r) ↔ ¬r ∨ p ∨ q"

set [set [Pos "p"; Pos "q"; Neg "r"]; set [Pos "q"; Pos "r"; Neg "p"]]

## Auxiliary Functions

The functions defined below make it convenient to create the propositional variables $\texttt{P}i$, $\texttt{T}i$, and $\texttt{Z}i$ for $i \in \{1,\cdots,9\}$:
- $\texttt{P}i$ is true iff the princess is in room number $i$,
- $\texttt{T}i$ is true iff there is a tiger in room number $i$,
- $\texttt{Z}i$ is true iff the inscription on room number $i$ is true.

Besides these variables we will also use the propositional variable `e`.  This variable will be `true` if the inscriptions on all empty rooms are true. 

The function `P` takes one argument:
- `i` is the number of a room.

It returns the string `Pi`.  The functions `T` and `Z` work in the same way.

In [5]:
let P (i: int) : string = sprintf "P%d" i

In [6]:
P 1

"P1"

In [7]:
parseKNF (sprintf "%s ∨ %s" (P 1) (P 2))

set [set [Pos "P1"; Pos "P2"]]

In [8]:
let T (i: int) : string = sprintf "T%d" i

In [9]:
let Z (i: int) : string = sprintf "Z%d" i

The function `atMostOne` takes one argument:
- `S` is a set of propositional variables.

It returns a set of clauses expressing the fact that at most one of the variables of $S$ is `true`.

In [10]:
let atMostOne (S: Set<string>) : CNF =
    set [ for p in S do
            for q in S do
                if p <> q then
                    set [ Neg p; Neg q ] ]

## Generating the Set of Clauses Describing the Problem

The function `computeClauses` takes no arguments.  It returns the set of clauses that encode the given problem.

In [11]:
let computeClauses () : CNF =
    // The princess has to be somewhere, i.e. there is a room containing the princess.
    let mutable Clauses : CNF = set [ set [ for i in 1 .. 9 -> Pos (P i) ] ]
    // There is just one princess.
    Clauses <- Clauses + atMostOne (set [ for i in 1 .. 9 -> P i ])
    for i in 1 .. 9 do
        // In the room containing the princess, the label at the door is true.
        Clauses <- Clauses + parseKNF (sprintf "%s → %s" (P i) (Z i))
        // In those rooms where there are tigers, the label is false.
        Clauses <- Clauses + parseKNF (sprintf "%s → ¬%s" (T i) (Z i))
        // The variable "e" is true if the labels of ALL empty rooms are true.
        // Either all labels of empty rooms are true or all those labels are false.
        Clauses <- Clauses + parseKNF (sprintf "(¬%s ∧ ¬%s) → (%s ↔ e)" (P i) (T i) (Z i))
    // Room Nr.1: The princess is in a room with an odd room number.
    //            The rooms with even numbers do not have tigers.
    Clauses <- Clauses + parseKNF "Z1 ↔ (P1 ∨ P3 ∨ P5 ∨ P7 ∨ P9) ∧ ¬T2 ∧ ¬T4 ∧ ¬T6 ∧ ¬T8"
    // Room Nr.2: This room is empty.
    Clauses <- Clauses + parseKNF "Z2 ↔ ¬P2 ∧ ¬T2"
    // Room Nr.3: The label at room number 5 is true, the label at room number 7 is false 
    //            and there is a tiger in room number 3
    Clauses <- Clauses + parseKNF "Z3 ↔ Z5 ∧ ¬Z7 ∧ T3"
    // Room Nr.4: The label at room number 1 is false, there is no tiger in room number 8
    //            and the label at room number 9 is true.
    Clauses <- Clauses + parseKNF "Z4 ↔ ¬Z1 ∧ ¬T8 ∧ Z9"
    // Room Nr.5: If the label at room number 2 or room number 4 is true, 
    //            then there is no tiger in room number 1.
    Clauses <- Clauses + parseKNF "Z5 ↔ (Z2 ∨ Z4) → ¬T1"
    // Room Nr.6: The label on room number 3 is false, the princess is in room number 2
    //            and there is no tiger in room number 2.
    Clauses <- Clauses + parseKNF "Z6 ↔ ¬Z3 ∧ P2 ∧ ¬T2"
    // Room Nr.7: The princess is in room number 1 and the label of room number 5 is true.
    Clauses <- Clauses + parseKNF "Z7 ↔ P1 ∧ Z5"
    // Room Nr.8: There is no tiger in this room and room number 9 is empty.
    Clauses <- Clauses + parseKNF "Z8 ↔ ¬T8 ∧ ¬P9 ∧ ¬T9"
    // Room Nr.9: Neither this room nor room number 1 has a tiger and 
    //            the label of room number 6 is true.
    Clauses <- Clauses + parseKNF "Z9 ↔ ¬T9 ∧ ¬T1 ∧ Z6"
    Clauses

In [12]:
let Clauses = computeClauses ()
printfn "%s" (cnfToString Clauses)

{{P1, P2, P3, P4, P5, P6, P7, P8, P9}, {P1, P3, P5, P7, P9, ¬Z1}, {P1, T1, Z1, ¬e}, {P1, T1, e, ¬Z1}, {P1, ¬Z7}, {P2, T2, Z2}, {P2, T2, Z2, ¬e}, {P2, T2, e, ¬Z2}, {P2, ¬Z6}, {P3, T3, Z3, ¬e}, {P3, T3, e, ¬Z3}, {P4, T4, Z4, ¬e}, {P4, T4, e, ¬Z4}, {P5, T5, Z5, ¬e}, {P5, T5, e, ¬Z5}, {P6, T6, Z6, ¬e}, {P6, T6, e, ¬Z6}, {P7, T7, Z7, ¬e}, {P7, T7, e, ¬Z7}, {P8, T8, Z8, ¬e}, {P8, T8, e, ¬Z8}, {P9, T8, T9, Z8}, {P9, T9, Z9, ¬e}, {P9, T9, e, ¬Z9}, {T1, T9, Z9, ¬Z6}, {T1, Z5}, {T2, T4, T6, T8, Z1, ¬P1}, {T2, T4, T6, T8, Z1, ¬P3}, {T2, T4, T6, T8, Z1, ¬P5}, {T2, T4, T6, T8, Z1, ¬P7}, {T2, T4, T6, T8, Z1, ¬P9}, {T2, Z3, Z6, ¬P2}, {T3, ¬Z3}, {T8, Z1, Z4, ¬Z9}, {Z1, ¬P1}, {Z2, Z4, Z5}, {Z2, ¬P2}, {Z3, Z7, ¬T3, ¬Z5}, {Z3, ¬P3}, {Z4, ¬P4}, {Z5, ¬P5}, {Z5, ¬Z3}, {Z5, ¬Z7}, {Z6, ¬P6}, {Z6, ¬Z9}, {Z7, ¬P1, ¬Z5}, {Z7, ¬P7}, {Z8, ¬P8}, {Z9, ¬P9}, {Z9, ¬Z4}, {¬P1, ¬P2}, {¬P1, ¬P3}, {¬P1, ¬P4}, {¬P1, ¬P5}, {¬P1, ¬P6}, {¬P1, ¬P7}, {¬P1, ¬P8}, {¬P1, ¬P9}, {¬P2, ¬P3}, {¬P2, ¬P4}, {¬P2, ¬P5}, {¬P2, ¬P6}, {¬P2, 

There are 110 clauses.

In [13]:
Clauses.Count

110

Finally, we call the function `solve` from the notebook `07-Davis-Putnam-JW.ipynb` to solve the problem.

In [14]:
#!time
let solution = solve Clauses

Wall time: 13.0217ms

In [15]:
solution

set [set [Pos "P5"]; set [Pos "T2"]; set [Pos "T8"]; set [Pos "Z5"]; set [Neg "P1"]; set [Neg "P2"]; set [Neg "P3"]; set [Neg "P4"]; set [Neg "P6"]; set [Neg "P7"]; set [Neg "P8"]; set [Neg "P9"]; set [Neg "T3"]; set [Neg "T5"]; set [Neg "Z1"]; set [Neg "Z2"]; set [Neg "Z3"]; set [Neg "Z4"]; set [Neg "Z6"]; set [Neg "Z7"]; set [Neg "Z8"]; set [Neg "Z9"]; set [Neg "e"]]

The function `getSolution` takes one argument:
- `S` is a set of unit clauses representing the solution of the problem.

It returns the variable `Pi` that is true in the solution, i.e. the variable specifying the room where the princess is located.

**Implementation:** `Seq.pick` applies the given function to the elements of a sequence until the function returns a value of the form `Some x`.  Then `x` is returned.  Here, we look for a unit clause containing a positive literal whose variable starts with `P`.

In [16]:
let getSolution (S: CNF) : string =
    S |> Seq.pick (fun Unit ->
        match arb Unit with
        | Pos v when v.StartsWith "P" -> Some v
        | _                           -> None)

We print the solution.

In [17]:
let princess = getSolution solution
princess

"P5"

In [18]:
printfn "Die Prinzessin ist im Zimmer Nummer %c." princess[1]

Die Prinzessin ist im Zimmer Nummer 

5

.

Finally, we check whether the solution is unique.  If the solution is not unique, then you have missed to code some of the requirements.

The function `checkUniqueness` takes two arguments:
- `Clauses` is the set of clauses describing the problem,
- `princess` is the variable specifying the room of the princess.

It adds the unit clause $\{\neg\texttt{princess}\}$ to `Clauses`.  If the resulting set of clauses is unsatisfiable, the room of the princess is uniquely determined.

In [19]:
let checkUniqueness (Clauses: CNF) (princess: string) : unit =
    let alternative = solve (Clauses.Add (set [ Neg princess ]))
    if alternative = set [ Set.empty ] then
        printfn "The solution is unique."
    else
        printfn "ERROR: The solution is not unique."

In [20]:
checkUniqueness Clauses princess

The solution is unique.